# deepMaze — train DRQN + DTQN on Colab (Drive-backed)

Mounts Google Drive, clones the repo into `/content/deepMaze`, trains **DRQN** then **DTQN** on a 30×60 multi-treasure + lava maze, and persists everything (MLflow runs, model weights, bundles, replays) to Drive at `${DRIVE_BASE}`.

**No GCP required** — MLflow uses a `file://` tracking store on Drive.

After the run finishes:
- `${DRIVE_BASE}/mlruns/` — full MLflow experiment store (open with `mlflow ui --backend-store-uri ${DRIVE_BASE}/mlruns`)
- `${DRIVE_BASE}/assets/<run_name>/` — ready-to-inference bundles (`config.json` + `model.pt` + `viz/replay.webp`). Copy to your local `assets/` to use with `python web/server.py`.

> **Heads up:** 30×60 with `partial_view=3` is hard — the agent has a 7×7 window in a ~1800-cell maze. Training takes ~1–2 h on T4 for DRQN at 6 k episodes. Drop `MAZE_WIDTH/HEIGHT` for faster smoke-tests.


## 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


## 2 — Config (Colab form)

In [ ]:
# @title Config { run: "auto" }
REPO_URL    = "https://github.com/juan-garassino/deepMaze.git"  # @param {type:"string"}
REPO_BRANCH = "main"                                             # @param {type:"string"}

# Where Drive-backed outputs live. mlruns/ and assets/ get created under this.
DRIVE_BASE = "/content/drive/MyDrive/deepMaze"                   # @param {type:"string"}

# Which agents to train, in order. Comma-separated.
AGENTS_TO_RUN = "drqn,dtqn"                                      # @param {type:"string"}
RUN_TAG       = "v1"                                             # @param {type:"string"}

# Maze config — shared across all agents in this notebook run.
MAZE_WIDTH   = 30       # @param {type:"integer"}
MAZE_HEIGHT  = 60       # @param {type:"integer"}
GENERATOR    = "dfs"    # @param ["open", "dfs", "random"]
DENSITY      = 0.2      # @param {type:"number"}
N_TREASURES  = 3        # @param {type:"integer"}
N_LAVA       = 2        # @param {type:"integer"}
COLLECT_ALL  = True     # @param {type:"boolean"}
PARTIAL      = 3        # @param {type:"integer"}

# Training budget. Scaled for 30x60 — shortest path can be ~100 steps,
# multi-treasure traversal pushes that to several hundred, so MAX_STEPS=1500
# leaves headroom. DTQN is heavier per step; same episode count is fine.
NUM_EPISODES  = 6000    # @param {type:"integer"}
MAX_STEPS     = 1500    # @param {type:"integer"}
EVAL_EPISODES = 50      # @param {type:"integer"}
SEED          = 0       # @param {type:"integer"}

MLFLOW_EXPERIMENT = "deepmaze"  # @param {type:"string"}


## 3 — Clone repo + install deps

In [ ]:
import os
import pathlib
import subprocess
import sys

WORK = pathlib.Path("/content/deepMaze")
if WORK.exists():
    subprocess.check_call(["git", "-C", str(WORK), "fetch", "--depth=1", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(WORK), "checkout", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(WORK), "reset", "--hard", f"origin/{REPO_BRANCH}"])
else:
    subprocess.check_call(["git", "clone", "--depth=1", "-b", REPO_BRANCH, REPO_URL, str(WORK)])

os.chdir(WORK)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", "requirements.txt", "mlflow==2.18.*"])

for sub in ("agents", "environment", "training", "utils", "config"):
    p = str(WORK / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

print("repo:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip())


## 4 — Drive paths + MLflow tracking URI

In [ ]:
from pathlib import Path

DRIVE_BASE_P = Path(DRIVE_BASE)
MLRUNS_DIR   = DRIVE_BASE_P / "mlruns"
ASSETS_DIR   = DRIVE_BASE_P / "assets"

MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_TRACKING_URI = f"file://{MLRUNS_DIR.as_posix()}"

import mlflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
print("tracking:", MLFLOW_TRACKING_URI)
print("assets  :", ASSETS_DIR)


## 5 — Train one agent (function)

In [ ]:
import json
import shutil
import time
from pathlib import Path

import mlflow
import torch

from maze import MazeEnvironment, RenderMaze
from recorders import ReplayRecorder
from seeding import seed_everything
from train import create_agent, evaluate_agent, simulate_episode, train_agent
from viz_events import EpisodeEvent, EventBus

PRINT_EVERY = 50  # episodes between progress lines


def train_one(agent_type: str, run_name: str) -> dict:
    print(f"\n=== {agent_type.upper()} — {run_name} ===")
    seed_everything(SEED)

    env = MazeEnvironment(
        width=MAZE_WIDTH, height=MAZE_HEIGHT, density=DENSITY,
        generator=GENERATOR, n_lava=N_LAVA, n_treasures=N_TREASURES,
        collect_all=COLLECT_ALL, partial_view=PARTIAL, seed=SEED,
    )
    agent = create_agent(agent_type, env)
    print("agent:", type(agent).__name__, "  episodes:", NUM_EPISODES, "  max_steps:", MAX_STEPS)

    bus = EventBus()

    def _on_ep_mlflow(ev: EpisodeEvent):
        mlflow.log_metrics(
            {"episode_reward": ev.total_reward,
             "episode_length": ev.length,
             "epsilon": ev.epsilon},
            step=ev.episode,
        )

    _state = {"reward_ema": None, "t0": time.time()}

    def _on_ep_print(ev: EpisodeEvent):
        r = ev.total_reward
        _state["reward_ema"] = r if _state["reward_ema"] is None else 0.9 * _state["reward_ema"] + 0.1 * r
        if ev.episode % PRINT_EVERY == 0 or ev.episode == NUM_EPISODES:
            elapsed = time.time() - _state["t0"]
            eps_per_s = (ev.episode + 1) / max(elapsed, 1e-6)
            remaining = (NUM_EPISODES - ev.episode - 1) / max(eps_per_s, 1e-6)
            extra = f"  loss={ev.loss:.4f}" if ev.loss is not None else ""
            print(f"ep {ev.episode:>5}/{NUM_EPISODES}  "
                  f"R={r:+.2f}  ema={_state['reward_ema']:+.2f}  "
                  f"len={ev.length:>4}  eps={ev.epsilon:.3f}{extra}  "
                  f"  [{eps_per_s:.1f} ep/s  ETA {remaining/60:.1f} min]",
                  flush=True)

    bus.subscribe(_on_ep_mlflow)
    bus.subscribe(_on_ep_print)

    params = dict(
        agent_type=agent_type, run_name=run_name,
        maze_width=MAZE_WIDTH, maze_height=MAZE_HEIGHT,
        generator=GENERATOR, density=DENSITY,
        n_treasures=N_TREASURES, n_lava=N_LAVA, collect_all=COLLECT_ALL,
        partial=PARTIAL, num_episodes=NUM_EPISODES, max_steps=MAX_STEPS,
        eval_episodes=EVAL_EPISODES, seed=SEED,
    )

    t0 = time.time()
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(params)
        train_agent(env, agent, num_episodes=NUM_EPISODES, max_steps=MAX_STEPS, bus=bus)
        mean_reward, mean_length, success_rate = evaluate_agent(
            env, agent, num_episodes=EVAL_EPISODES, max_steps=MAX_STEPS,
        )
        mlflow.log_metrics({
            "eval_mean_reward": mean_reward,
            "eval_mean_length": mean_length,
            "eval_success_rate": success_rate,
        })
        run_id = run.info.run_id
    print(f"trained in {time.time() - t0:.1f}s — eval success: {success_rate:.2%}")

    # --- Bundle: write to BOTH local /content (for in-Colab smoke) AND Drive (persistent) ---
    local_out = Path(f"assets/{run_name}")
    drive_out = ASSETS_DIR / run_name
    for out in (local_out, drive_out):
        (out / "viz").mkdir(parents=True, exist_ok=True)

    config = dict(
        agent_type=agent_type, net=None,
        maze_width=MAZE_WIDTH, maze_height=MAZE_HEIGHT,
        n_agents=1, density=DENSITY, generator=GENERATOR,
        no_ensure_solvable=False, n_lava=N_LAVA, lava_reward=-1.0,
        partial=PARTIAL, n_treasures=N_TREASURES, collect_all=COLLECT_ALL,
        seed=SEED, num_episodes=NUM_EPISODES, max_steps=MAX_STEPS,
        eval_episodes=EVAL_EPISODES, learning_rate=None, discount_factor=0.99,
        image_path=None, sprite_files=["sprites.png"], sprite_size=32,
        replay_fmt="webp", frame_skip=1, max_frames=None,
        policy_snapshot_every=50, live=False, live_web=False, web_port=8000,
        run_id=run_id, run_name=run_name,
        random_start=False, resume=None, eval_maze="same", eval_seeds=1,
    )
    (local_out / "config.json").write_text(json.dumps(config, indent=2))

    module = getattr(agent, "model", None) or getattr(agent, "ac", None)
    torch.save(module.state_dict(), local_out / "model.pt")

    agent.set_deterministic(True)
    try:
        _, positions, _, _ = simulate_episode(env, agent, max_steps=MAX_STEPS, at_start=True)
    finally:
        agent.set_deterministic(False)

    full_frames = [env.maze.copy() for _ in positions]
    sprites = RenderMaze.placeholder_sprites(32)
    rm = RenderMaze(sprites)
    ReplayRecorder(rm).feed(full_frames, positions, None)
    rm.save(str(local_out / "viz" / "replay.webp"), fmt="webp")

    # Mirror local → drive
    if drive_out.exists():
        shutil.rmtree(drive_out)
    shutil.copytree(local_out, drive_out)

    # Log artifacts back to MLflow (still file://Drive, so this is in-Drive too)
    with mlflow.start_run(run_id=run_id):
        mlflow.log_artifacts(str(local_out), artifact_path=f"assets/{run_name}")

    print("bundle:", drive_out)
    return {
        "agent_type": agent_type,
        "run_name": run_name,
        "run_id": run_id,
        "eval_success_rate": success_rate,
        "eval_mean_reward": mean_reward,
        "drive_path": str(drive_out),
    }


## 6 — Train DRQN then DTQN sequentially

In [ ]:
agents = [a.strip() for a in AGENTS_TO_RUN.split(",") if a.strip()]
results = []
for agent_type in agents:
    run_name = f"{agent_type}_{RUN_TAG}"
    results.append(train_one(agent_type, run_name))

print("\n=== summary ===")
for r in results:
    print(f"  {r['agent_type']:5s}  {r['run_name']:20s}  success={r['eval_success_rate']:.2%}  → {r['drive_path']}")


## 7 — Use the bundles locally

On your laptop:

```bash
# Copy a bundle from Drive into the repo's assets/
rsync -a "<google-drive-folder>/deepMaze/assets/drqn_v1/" assets/drqn_v1/

# Then either:
python web/server.py --port 8000    # → http://localhost:8000  → pretrained dropdown
# or:
python main.py --agent_type drqn --resume assets/drqn_v1/model.pt --num_episodes 0
```

To browse MLflow runs locally after copying `mlruns/` down from Drive:

```bash
mlflow ui --backend-store-uri "file://$(pwd)/mlruns"
```
